In [ ]:
from functions import find_f
import numpy as np


def pca(givens:dict,model:str = 'TURBOJET',SI:bool =True)->dict:
    """  
    This function performs Parametric Cycle Analysis for a real engine using the algorithms 
    described in Chapter 7 of Mattingly. 
    
    Args:
        givens: A dict containing all the necessary values, given or assumed. See Mattingly for required info
        model: String dictating which algorithm to follow (TURBOJET or TURBOFAN)
        SI: bool for what type of units, defaults to True (meaning SI). Also assumes then in kJ/kg per table entries

    Returns:
        solution: dict containing the helpful values to return
    """

    if model.upper() == 'TURBOJET':
        M0,T0,gac,cpc,gat,cpt,hpr,pid_max,pib,pin,ec,et,etab,etam,P0P9,Tt4,pic = givens.items()
        M0 = M0[1]
        T0 = T0[1]
        gac = gac[1]
        cpc = cpc[1]
        gat = gat[1]
        cpt = cpt[1]
        hpr = hpr[1]
        pid_max = pid_max[1]
        pib = pib[1]
        pin = pin[1]
        ec = ec[1]
        et = et[1]
        etab = etab[1]
        etam = etam[1]
        P0P9 = P0P9[1]
        Tt4 = Tt4[1]
        pic = pic[1]

        # Defining gc
        if SI:
            gc = 1
        else:
            gc = 32.174

        print('Freestream and Ram Properties')
        Rc = (gac - 1)/gac*cpc
        Rt = (gat - 1)/gat*cpt
        if SI:
            a0 = np.sqrt(gac*Rc*gc*T0*1000)
        V0 = a0*M0
        taur = 1 + (gac-1)/2*M0**2
        pir = taur**(gac/(gac-1))
        # Finding eta_r
        if M0<1:
            etar = 1
        elif M0>1 and M0<5:
            etar = 1 - 0.075*(M0 - 1)**1.35
        else:
            etar = 800/(M0**4 + 935)
        print(f'    R_c = {round(Rc,4)} kJ/kgK, R_t = {round(Rt,4)} kJ/kgK')
        print(f'    a_0 = {round(a0,4)} m/s, V_0 = {round(V0,4)} m/s')
        print(f'    tau_r = {round(taur,4)}, pi_r = {round(pir,4)}, eta_r = {round(etar,4)}')

        print('Inlet Diffuser')
        pid = pid_max*etar
        taul = cpt*Tt4/(cpc*T0)
        print(f'    pi_d = {round(pid,4)}, tau_lambda = {round(taul,4)}')

        print('Compressor')
        tauc = pic**((gac - 1)/(gac*ec))
        etac = (pic**((gac-1)/gac)-1)/(tauc - 1)
        Tt3 = T0*taur*tauc
        print(f'    tau_c = {round(tauc,4)}, eta_c = {round(etac,4)}, Tt3 = {round(Tt3,4)}')

        print('Iterating to find ht4 and f')
        f,ht3,ht4 = find_f(Tt3,Tt4,etab,hpr,pr = True)
        print(f'    Final Values: f = {round(float(f),6)}, ht4 = {round(float(ht4),4)} kJ/kg')

        print('Turbine Properties')
        taut = 1 - 1/(etam*(1+f))*taur/taul*(tauc-1)
        pit = taut**(gat/((gat-1)*et))
        etat = (1 - taut)/(1-taut**(1/et))
        print(f'    tau_t = {round(taut,4)}, pi_t = {round(pit,4)}, eta_t = {round(etat, 4)}')

        print('Exit Properties')
        Pt9P9 = P0P9*pir*pid*pic*pib*pit*pin
        M9 = np.sqrt(2/(gat-1)*(Pt9P9**((gat-1)/gat)-1))
        T9T0 = Tt4*taut/T0/(Pt9P9**((gat-1)/gat))
        T9 = T9T0*T0
        V9a0 = M9*np.sqrt(gat*Rt*T9/(gac*Rc*T0))
        print(f'    Pt9/P9 = {round(Pt9P9,4)}, T9 = {round(T9,4)} K, T9/T0 = {round(T9T0,4)}')
        print(f'    M9 = {round(M9,4)}, V9/a0 = {round(V9a0,4)}')

        print('Performance Properties')
        Fm0 = a0/gc*((1+f)*V9a0 - M0 + (1+f)*Rt*T9T0/(Rc*V9a0)*(1-P0P9)/gac)
        S = f/Fm0*1e6
        etaTH = a0**2*((1+f)*V9a0**2 - M0**2)/(2*gc*f*hpr)/1000
        etap = 2*gc*V0*Fm0/(a0**2*((1+f)*V9a0**2 - M0**2))
        etaO = etaTH*etap
        print(f'    F/m0 = {round(Fm0,4)} N/kg/s, S = {round(S,4)} mg/s/kN')
        print(f'    eta_TH = {round(etaTH,4)}, eta_p = {round(etap,4)}, eta_O = {round(etaO,4)}')

        solution = {"Rc":Rc,"Rt":Rt,"a0":a0,"V0":V0,"taur":taur,"pir":pir,"etar":etar,"pid":pid,
                    "taul":taul,"tauc":tauc,"etac":etac,"Tt3":Tt3,"f":f,"ht3":ht3,"taut":taut,
                    "pit":pit,"etat":etat,"Pt9P9":Pt9P9,"M9":M9,"T9T0":T9T0,"V9a0":V9a0,"Fm0":Fm0,
                    "S":S,"etaTH":etaTH,"etap":etap,"etaO":etaO}
        return solution





givens_jet = {"M0":2,"T0":216.7,"gac":1.4,"cpc":1.004,"gat":1.3,"cpt":1.239,"hpr":42800,"pid_max":0.95,
          "pib":0.94,"pin":0.96,"ec":0.9,"et":0.9,"etab":0.98,"etam":0.99,"P0P9":0.5,"Tt4":1800,"pic":10}
pca(givens_jet,'TURBOJET')


Freestream and Ram Properties
    R_c = 0.2869 kJ/kgK, R_t = 0.2859 kJ/kgK
    a_0 = 295.0029 m/s, V_0 = 590.0058 m/s
    tau_r = 1.8, pi_r = 7.8244, eta_r = 0.925
Inlet Diffuser
    pi_d = 0.8788, tau_lambda = 10.2506
Compressor
    tau_c = 2.0771, eta_c = 0.8641, Tt3 = 810.1991
Iterating to find ht4 and f
    f = 0.03060681182906731, ht4 = 2053.83, n = 0
    f = 0.03164401048318941, ht4 = 2093.934949203673, n = 1
    f = 0.03172139747739065, ht4 = 2096.9240088088586, n = 2
    f = 0.03172716617331017, ht4 = 2097.1468057577144, n = 3
    f = 0.03172759616308055, ht4 = 2097.1634126017293, n = 4
    Final Values: f = 0.031728, ht4 = 2097.1634 kJ/kg
Turbine Properties
    tau_t = 0.8148, pi_t = 0.3731, eta_t = 0.9099
Exit Properties
    Pt9/P9 = 11.5739, T9 = 833.5248 K, T9/T0 = 3.8464
    M9 = 2.2504, V9/a0 = 4.246
Performance Properties
    F/m0 = 800.4723 N/kg/s, S = 39.6361 mg/s/kN
    eta_TH = 0.4679, eta_p = 0.7434, eta_O = 0.3478
